# Lecture 07 — Time-series forecasting: information, origins, and honest evaluation

We forecast synthetic daily sales from their past and from calendar values known in advance. The central mathematical object is a conditional expectation given the information available at a forecast origin. We will compare a seasonal-naive rule with an XGBoost model, and evaluate both in a rolling one-step-ahead setting.

![Teaching figure for time series](../../docs/figures/07_time_series.svg)

*The supplied synthetic series with illustrative 70/15/15 boundaries on raw rows. The notebook splits usable lagged rows, so its exact boundaries differ.*

## Translate the information set into a forecast row

We `predict` the sales for date $t$ after observing date $t-1$. For example, to `predict` January 9 after January 8 closes, `lag_1` is January 8 sales and `lag_7` is January 2 sales. The seven-day mean uses January 2 through January 8. January 9 sales must appear only as the target, never inside those features.

`shift(1)` moves each observed value into the next row, aligning yesterday's observation with today's target. A `rolling` mean without this `shift` would include today's target and leak it into the inputs. Calendar values, such as the weekday for January 9, are known in advance.

During `rolling` one-step test evaluation, a completed test day's actual sales become available for the next day's lag features. The model parameters remain fixed. Predicting an entire future month before observing any of it is a different problem. The expanding-window section additionally refits models at several development origins; follow its input and output partitions carefully.

## 1. Conditional expectation is the optimal squared-error forecast

Let $(\Omega,\mathcal A,\mathbb P)$ be a probability space, assume $Y_{t+h}\in L^2(\Omega,\mathcal A,\mathbb P)$, and let $Y_{t+h}$ be the value to forecast at origin $t$ and horizon $h$. The information available at the origin is a sigma-algebra $\mathcal F_t\subseteq\mathcal A$. For any $g\in L^2(\Omega,\mathcal A,\mathbb P)$ that is $\mathcal F_t$-measurable, condition on the information:

$$\mathbb E[(Y_{t+h}-g)^2\mid\mathcal F_t]
=\operatorname{Var}(Y_{t+h}\mid\mathcal F_t)
+\left(g-\mathbb E[Y_{t+h}\mid\mathcal F_t]\right)^2.$$

Thus the minimum-risk forecast is the conditional mean $\hat Y_{t+h\mid t}=\mathbb E[Y_{t+h}\mid\mathcal F_t]$ (up to almost-sure equality). A forecast is a function of what is known at its origin. If a predictor uses a realized value from after $t$, it is not $\mathcal F_t$-measurable and does not describe a forecast that could have been issued then.

For absolute-error loss, a conditional median is optimal. We will report MAE and RMSE: both are interpretable in sales units, and RMSE gives larger misses more weight. We do not use MAPE because its percentage errors behave poorly near a zero response.

## 2. A time series has ordered information

The filtration $\mathcal F_0\subseteq\mathcal F_1\subseteq\cdots$ represents information accumulating through time. Here it includes past observed sales $Y_s$ for $s\leq t$ and calendar values known for future dates. It excludes future sales.

An autoregression of order $p$ can use a lag window:

$$\hat Y_{t+1\mid t}=f(Y_t,Y_{t-1},\ldots,Y_{t-p+1},\text{calendar}_{t+1}).$$

A lag is a value selected by its distance from the target. A window is a collection of past values summarized together, for example a `rolling` mean. The 7-day seasonal-naive rule is $\hat Y_{t+1\mid t}=Y_{t-6}$, the latest observed value for the same weekday. It is a transparent baseline, not a fitted model.

### Horizon and gap answer different questions

The forecast **horizon** $h$ is how far ahead the target lies from the forecast origin. A **gap** is a deliberate block of observations omitted between the last training target and the first validation or test target. They are not interchangeable. This lesson uses horizon $h=1$ and adjacent chronological blocks, so its split gap is zero. For a horizon-$h$ target, features at origin $t$ `predict` $Y_{t+h}$; the information set still ends at $t$. If the application cannot refresh recent labels, or target windows overlap, choose a gap/purge rule that matches that delay and target construction.

| Validation design | What it simulates | Main choice |
|---|---|---|
| One chronological holdout | One later deployment period | A single fixed cutoff; simple final assessment |
| Expanding window | Refit as history grows | Keep all earlier observations; training cost grows |
| Rolling window | Refit using only recent history | Set a history width to adapt to drift; older data is discarded |
| Gap or purge | Delayed labels or overlapping target windows | Exclude boundary rows whose labels would not yet be known |

Random row splitting breaks the information order. Chronological evaluation estimates the future-facing question, but the dates and update schedule must match the intended use.

## 3. Why an iid random split answers the wrong question

A random split usually allows training and test rows to be interleaved. Adjacent dates can be nearly duplicates in their history, and a model can train on dates later than a test date. This estimates performance on randomly hidden rows under an exchangeability assumption. Forecasting asks about future dates from an earlier information set, so we hold out contiguous future blocks and preserve their order.

Our primary scenario is explicit: at each test date $t$, issue a one-day-ahead forecast, then observe $Y_t$ before issuing the next forecast. The model parameters remain fixed after training, but the lag features update with each newly observed actual. This is a rolling-origin one-step evaluation. A fixed-origin 30-day forecast would have to recursively feed its own predictions into future lags (or use a direct horizon-specific model); it cannot use actual sales from within those 30 days.

## 4. Generate features without looking forward

For the preceding seven complete observations, the rolling mean on target row $t$ is
$$\overline Y_{t,7}=\frac{1}{7}\sum_{j=1}^{7}Y_{t-j}.$$
This summarizes a local level; `lag_7` separately preserves the observation from the matching weekday.

For target date $t$, `shift(k)` aligns the observation $Y_{t-k}$ with row $t$. A rolling window summarizes trailing rows; it does not automatically know which values were available at the forecast origin. Shift the target series first, then roll it. For example, `sales.shift(1).rolling(window=7, min_periods=7).mean()` uses the seven rows ending at $t-1$ and requires all seven values. Without `shift(1)`, the current target can enter its own predictor.

An integer window such as `window=7` means seven observations. A time-offset window such as `7D` on a datetime index means a seven-day interval; with missing dates, those meanings differ. `min_periods` controls how many nonmissing values are needed for a result. Requiring a full window avoids silently changing the summary's sample size across dates. A smaller minimum keeps early rows but makes early averages less comparable. A centered window is unsuitable for forecasting because it can extend beyond the origin.

| Pandas choice | Meaning here | When to change it |
|---|---|---|
| `shift(k)` | Put the value from $k$ rows earlier on today's target row | Change `k` for another lag; confirm row spacing first |
| `rolling(window=7)` | Summarize the trailing seven observations | Use an offset window for elapsed calendar time |
| `min_periods=7` | Return a value only with seven observations | Lower it only when partial windows match the feature definition |
| `mean()` | Average the available values in the window | Use `sum()`, `std()`, `min()`, or another stated summary |

Calendar features are deterministic functions of the target date and known in advance. This fixture is a pedagogical daily simulation with a fixed seed; its generator is documented in `sales_dataset.md`. The loader searches upward from either the repository root or the lecture folder.

Pandas references: [Series.shift](https://pandas.pydata.org/docs/reference/api/pandas.Series.shift.html) and [Series.rolling](https://pandas.pydata.org/docs/reference/api/pandas.Series.rolling.html) document alignment and window options.

### Load and check the time axis

The next cell locates the committed CSV, parses date as timestamps, sorts rows, and checks that dates are unique and increasing. These checks protect alignment: shifting an unsorted series would associate the wrong observation with each target.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_data_file(filename):
    """Search this directory and its parents for a file under data/."""
    for directory in (Path.cwd(), *Path.cwd().parents):
        candidate = directory / "data" / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find data/{filename} from {Path.cwd()}")


sales_data = pd.read_csv(find_data_file("daily_sales.csv"), parse_dates=["date"])
sales_data = sales_data.sort_values("date").reset_index(drop=True)
assert sales_data["date"].is_monotonic_increasing and sales_data["date"].is_unique
sales_data.head()

### See how shifting and `min_periods` change the rows

This small example makes alignment visible before applying it to the full fixture. Each shifted summary uses only earlier observations. With `min_periods=1`, the first averages use shorter histories; with `min_periods=3`, they appear only after three observations.

In [ ]:
toy = pd.Series([10.0, 12.0, 11.0, 15.0, 14.0],
                  index=pd.date_range("2025-01-01", periods=5, freq="D"), name="sales")
toy_features = pd.DataFrame({
    "target_sales": toy,
    "lag_1": toy.shift(1),
    "mean_3_full": toy.shift(1).rolling(window=3, min_periods=3).mean(),
    "mean_3_partial": toy.shift(1).rolling(window=3, min_periods=1).mean(),
})
toy_features

In [ ]:
def make_forecasting_features(data):
    """Build rolling one-step predictors using only observations before each date."""
    frame = data[["date", "sales"]].copy().sort_values("date").reset_index(drop=True)
    day = frame["date"].dt.dayofweek.to_numpy()
    frame["weekday_sin"] = np.sin(2 * np.pi * day / 7)
    frame["weekday_cos"] = np.cos(2 * np.pi * day / 7)
    frame["year_day_sin"] = np.sin(2 * np.pi * frame["date"].dt.dayofyear / 365.25)
    frame["year_day_cos"] = np.cos(2 * np.pi * frame["date"].dt.dayofyear / 365.25)
    for lag in (1, 2, 7, 14, 28):
        frame[f"lag_{lag}"] = frame["sales"].shift(lag)
    frame["mean_7_before"] = frame["sales"].shift(1).rolling(7).mean()
    frame["mean_28_before"] = frame["sales"].shift(1).rolling(28).mean()
    return frame.dropna().reset_index(drop=True)


feature_data = make_forecasting_features(sales_data)
feature_columns = [
    column for column in feature_data.columns if column not in ("date", "sales")
]
feature_data[["date", "sales", "lag_1", "lag_7", "mean_7_before", "weekday_sin"]].head()

### Build the feature matrix

The function first sorts and resets the row index, then computes sine/cosine calendar encodings so cyclical values meet smoothly at the year or week boundary. Each `shift(k)` supplies one lag. The rolling means begin with `shift(1)`, so the target date is excluded. dropna() removes rows without the full 28-observation history required by the largest lag and `rolling` window. `feature_columns` then defines input matrix X; sales remains target y, and date stays available for diagnostics.

### Five concrete forecast rows

The origin is the last observed date, one day before the target date. Each row below names the most recent available observation and the seven-day lag. Target sales are shown for checking alignment after the event, and are excluded from the predictor columns.

In [ ]:
worked_dates = feature_data.head(5).copy()
worked_dates.insert(0, "origin_date", worked_dates.date - pd.Timedelta(days=1))
worked_dates.insert(2, "lag_7_date", worked_dates.date - pd.Timedelta(days=7))
display(worked_dates[["origin_date", "date", "lag_7_date", "lag_1", "lag_7", "mean_7_before", "sales"]])

Verify the boundary: the first complete row has 28 past observations. `lag_1` is the preceding day's observed sales, while `mean_7_before` excludes the row's own target. The calendar fields are known for the forecast date.

In [ ]:
first_row = feature_data.iloc[0]
assert np.isclose(first_row["lag_1"], sales_data.loc[27, "sales"])
assert np.isclose(first_row["mean_7_before"], sales_data.loc[21:27, "sales"].mean())
assert len(feature_data) == len(sales_data) - 28
print("Feature boundary checks passed:", first_row["date"].date())

A useful leakage test is to perturb outcomes strictly after one origin and verify that the earlier origin's features do not change.

In [ ]:
origin_raw_index = 200
origin_feature_index = origin_raw_index - 28
future_perturbation = sales_data.copy()
future_perturbation.loc[origin_raw_index :, "sales"] += 10000
perturbed_features = make_forecasting_features(future_perturbation)
predictor_columns = [
    column for column in feature_data.columns if column not in ("date", "sales")
]
np.testing.assert_allclose(
    feature_data.loc[origin_feature_index, predictor_columns].to_numpy(dtype=float),
    perturbed_features.loc[origin_feature_index, predictor_columns].to_numpy(
        dtype=float
    ),
)
print("Changing the target at the forecast date and all future targets leaves its predictors unchanged.")

The perturbation check changes the target at the chosen forecast date and every later date. Features for that origin should stay identical because their inputs end on the preceding date. This is a focused check of temporal alignment; it does not prove every possible transformation is leakage-free.

## 5. Chronological train, validation, and test boundaries

We partition usable feature rows in date order: the earliest 70% for training, the next 15% for validation, and the last 15% for final test. These proportions are illustrative; the dates and forecast protocol matter more than a familiar percentage. Validation chooses the boosting-round count through early stopping. The test block stays untouched until the final score is calculated.

Because this is `rolling` one-step evaluation, a later test row may use an earlier test day's observed sales as a lag: that actual would have arrived by the later forecast origin. Model parameters remain fixed, and test targets never enter fitting or model selection. If labels arrive after a delay, or a multi-step target overlaps the validation boundary, add a gap/purge consistent with that operational delay. A chronological split prevents training on later targets; it does not by itself guarantee every feature would be available in a particular deployment.

In [ ]:
n_rows = len(feature_data)
train_end = int(0.70 * n_rows)
validation_end = int(0.85 * n_rows)
train_data = feature_data.iloc[:train_end].copy()
validation_data = feature_data.iloc[train_end:validation_end].copy()
test_data = feature_data.iloc[validation_end:].copy()

boundary_table = pd.DataFrame(
    {
        "partition": ["train", "validation", "test"],
        "rows": [len(train_data), len(validation_data), len(test_data)],
        "first_date": [
            part["date"].min().date()
            for part in (train_data, validation_data, test_data)
        ],
        "last_date": [
            part["date"].max().date()
            for part in (train_data, validation_data, test_data)
        ],
    }
)
boundary_table

`iloc` slices by row position, so the percentages preserve sorted chronology. The boundary table exposes date ranges and sample counts; inspect it before fitting. Since early feature rows were dropped, these boundaries are proportions of usable rows, not raw CSV rows.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 2.5))
ax.plot(sales_data["date"], sales_data["sales"], color="0.55", lw=1)
partitions = (
    (train_data, "#2563eb", "train"),
    (validation_data, "#f59e0b", "validation"),
    (test_data, "#dc2626", "test"),
)
for part, color, label in partitions:
    ax.axvspan(
        part["date"].min(), part["date"].max(), color=color, alpha=0.13, label=label
    )
ax.set(title="Chronological forecast partitions", ylabel="daily sales")
ax.legend(ncol=3, frameon=False)
fig.autofmt_xdate()
plt.show()

The shaded plot places those exact feature-row partitions against the original series. It is a visual check that training precedes validation and validation precedes test.

## 6. Expanding-window development backtest

A single boundary can make the result depend on one especially easy or difficult season. We can inspect several forecast origins that all lie before the final test boundary. At each origin, use an expanding training history, reserve the following contiguous block as an inner validation block for early stopping, then score the next contiguous block as rolling one-step forecasts. No final-test rows enter this backtest. Because the model is refit at each origin, it answers a different deployment question than the fixed-model final split above.

In [ ]:
# Optional extension: implementation is in src/course_ml/forecasting.py.
# Read the main chronological experiment first if you are new to forecasting.
import sys
sys.path.insert(0, str(find_data_file("daily_sales.csv").parent.parent / "src"))
from course_ml.forecasting import rolling_development_backtest

# Development ends where the final test begins; folds never inspect the final test block.
development_folds = rolling_development_backtest(
    feature_data,
    feature_columns,
    development_end=validation_end,
    initial_train_size=350,
    validation_size=70,
    test_size=70,
)
development_folds

This optional code imports the `rolling` development helper. Each fold grows its training prefix, selects a round count on the next validation block, then measures the following block. It shows whether performance changes across development origins; it does not replace the final untouched test.

The expanding backtest is a development diagnostic. Its fold-specific validation blocks choose iteration counts; its evaluation blocks compare chronological origins within development data. The final test remains a separate last block. Neither backtest changes the primary forecast scenario: all scores above are rolling one-step-ahead, with observed lag updates.

## 7. Choose model complexity on validation only

XGBoost adds trees sequentially to reduce the training objective. Here the objective is squared error, so each new tree attempts to correct the current residual pattern. Tree depth, leaf constraints, sampling, and regularization limit how finely the ensemble can fit the small synthetic sample. The values below are a conservative starting configuration, not universal defaults.

| Parameter | What it controls | Why this lesson sets it |
|---|---|---|
| `objective="reg:squarederror"` | Training loss for a continuous target | Matches squared-error fitting and RMSE monitoring |
| `n_estimators=500` | Upper bound on boosting rounds | Gives early stopping room to find a useful stopping point |
| `learning_rate=0.035` | Shrinkage per added tree | Small steps usually need more rounds and can regularize the fit |
| `max_depth=3` | Maximum tree depth | Limits interactions and leaf count on a small dataset |
| `min_child_weight=5` | Minimum Hessian mass in a child | Discourages leaves supported by very little data |
| `subsample=0.9` | Row fraction sampled for each tree | Adds mild sampling regularization |
| `colsample_bytree=0.9` | Feature fraction sampled for each tree | Adds mild feature regularization |
| `reg_lambda=5.0` | L2 penalty on leaf weights | Shrinks leaf updates toward zero |
| `early_stopping_rounds=25` | Stop after validation RMSE fails to improve | Uses validation to select the effective number of trees |
| `eval_metric="rmse"` | Metric observed for early stopping | Keeps the stopping signal aligned with squared-error emphasis |
| `random_state`, `n_jobs` | Sampling reproducibility and parallel workers | Makes this small run repeatable and bounded |

For a leaf $j$, let $G_j=\sum_{i\in j}g_i$ and $H_j=\sum_{i\in j}h_i$ be sums of first and second derivatives of the loss. With L2 leaf penalty $\lambda$, the second-order objective gives the leaf weight
$$w_j^*=-\frac{G_j}{H_j+\lambda}.$$
The learning rate scales the tree contribution when it is added to the ensemble. This explains why regularization and a smaller learning rate shrink updates, while depth and child constraints control which partitions can receive their own updates. See the [XGBoost parameter reference](https://xgboost.readthedocs.io/en/stable/parameter.html) and the course's [boosting derivation](../03_xgboost/lesson.ipynb).

The scikit-learn wrapper accepts `early_stopping_rounds` on estimator construction in the installed XGBoost version. `fit` receives only training rows plus the chronological validation set. The fitted estimator remains trained on the training partition; validation chooses the stopping iteration but is not added to it. Never use the final test block to tune these choices. A simple seasonal baseline can still win: weekly recurrence may already explain most predictable structure, the sample is small, or trees may not extrapolate level trends well. Report that result and investigate on development data.

The [XGBoost scikit-learn API](https://xgboost.readthedocs.io/en/stable/python/python_api.html) documents estimator arguments and fitting behavior.

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

X_train = train_data[feature_columns]
y_train = train_data["sales"]
X_validation = validation_data[feature_columns]
y_validation = validation_data["sales"]

model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=500,
    learning_rate=0.035,
    max_depth=3,
    min_child_weight=5,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=5.0,
    early_stopping_rounds=25,
    eval_metric="rmse",
    random_state=17,
    n_jobs=2,
)
model.fit(X_train, y_train, eval_set=[(X_validation, y_validation)], verbose=False)
selected_rounds = model.best_iteration + 1
selected_rounds

The code passes named columns and target values to `fit`; `eval_set` is used only to monitor RMSE. `best_iteration` + 1 converts XGBoost's zero-based index into a tree count. The fitted model's `predict` method then maps each feature row to a sales estimate.

## 8. Rolling one-step test evaluation

The 7-day seasonal-naive rule and XGBoost receive the same test origins and targets. For each test origin, the feature builder used only past observed sales, including actuals that would already have arrived in this rolling deployment scenario. MAE and RMSE are reported in synthetic sales units. A negative improvement over the baseline means the learned model did worse on that score.

In [ ]:
test_prediction = model.predict(test_data[feature_columns])
seasonal_prediction = test_data["lag_7"].to_numpy()
y_test = test_data["sales"].to_numpy()


def score_predictions(actual, predicted):
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": mean_squared_error(actual, predicted) ** 0.5,
    }


score_table = pd.DataFrame(
    {
        "seasonal naive": score_predictions(y_test, seasonal_prediction),
        "XGBoost": score_predictions(y_test, test_prediction),
    }
).T
score_table["MAE improvement vs naive"] = (
    score_table.loc["seasonal naive", "MAE"] - score_table["MAE"]
)
score_table["RMSE improvement vs naive"] = (
    score_table.loc["seasonal naive", "RMSE"] - score_table["RMSE"]
)
score_table

Both predictions and targets use the same final-test rows. MAE is the average absolute miss; RMSE is the square root of average squared miss and therefore weights large errors more heavily. Positive improvement means lower error than the seasonal rule; a negative value favors the baseline.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True, layout="constrained")
axes[0].plot(test_data["date"], y_test, label="actual", color="black", lw=1.4)
axes[0].plot(test_data["date"], seasonal_prediction, label="seasonal naive", alpha=0.8)
axes[0].plot(test_data["date"], test_prediction, label="XGBoost", alpha=0.8)
axes[0].set_ylabel("sales")
axes[0].legend(ncol=3, frameon=False)
residual = y_test - test_prediction
axes[1].scatter(test_data["date"], residual, s=12, alpha=0.75)
axes[1].axhline(0, color="black", lw=1)
axes[1].set(
    ylabel="actual - predicted", xlabel="test date", title="XGBoost one-step residuals"
)
fig.autofmt_xdate()
plt.show()

## Limits and interpretation

Synthetic series make the time ordering and leakage rules visible, but they do not establish a real retailer's demand process. The seasonal-naive baseline uses a weekly period chosen by construction. Calendar encodings assume the forecast date is known. MAE/RMSE summarize errors but hide date-specific failures; inspect the timeline and residuals. A useful operational evaluation must reproduce the forecast horizon, update schedule, and covariate availability of the intended deployment.

Tree ensembles `predict` through leaf values and do not automatically extrapolate a linear trend. A drifting level can make the seasonal-naive rule competitive or better. A baseline win is an informative result: report it, and investigate using development data rather than tuning against the final test.